In [1]:
import mlflow
import dagshub
dagshub.init(repo_owner='Manoj-Gujare', repo_name='Youtube-Comment-Analyzer-Chrome-Plugin', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/Manoj-Gujare/Youtube-Comment-Analyzer-Chrome-Plugin.mlflow")

Accessing as Manoj-Gujare

Initialized MLflow to track repo "Manoj-Gujare/Youtube-Comment-Analyzer-Chrome-Plugin"

Repository Manoj-Gujare/Youtube-Comment-Analyzer-Chrome-Plugin initialized!

In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='mlflow-artifacts:/5cbb706aa34f4459ae6927f7f64ade6e', creation_time=1731954661328, experiment_id='4', last_update_time=1731954661328, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna


In [4]:
df = pd.read_csv('reddit_preprocessing.csv').dropna()
df.shape

(36662, 2)

In [5]:
# Step 1: (Optional) Remapping - skipped since not strictly needed for Logistic Regression

# Step 2: Remove rows where the target labels (category) are NaN
df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3)  # Trigram
max_features = 1000  # Set max_features to 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLflow
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algorithm_comparison")

        # Log algorithm name as a parameter
        mlflow.log_param("algo_name", model_name)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log the model
        mlflow.sklearn.log_model(model, f"{model_name}_model")


# Step 6: Optuna objective function for Logistic Regression
def objective_logreg(trial):
    C = trial.suggest_float('C', 1e-4, 10.0, log=True)
    penalty = trial.suggest_categorical('penalty', ['l1', 'l2'])

    # LogisticRegression model setup with balanced class weight
    model = LogisticRegression(C=C, penalty=penalty, solver='liblinear', random_state=42)
    return accuracy_score(y_test, model.fit(X_train, y_train).predict(X_test))


# Step 7: Run Optuna for Logistic Regression, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_logreg, n_trials=30)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LogisticRegression(C=best_params['C'], penalty=best_params['penalty'], solver='liblinear', random_state=42)

    # Log the best model with MLflow, passing the algo_name as "LogisticRegression"
    log_mlflow("LogisticRegression", best_model, X_train, X_test, y_train, y_test)

# Run the experiment for Logistic Regression
run_optuna_experiment()


[I 2024-11-19 00:13:20,454] A new study created in memory with name: no-name-b6a88221-845c-44be-9072-ab164b68b293
[I 2024-11-19 00:13:20,720] Trial 0 finished with value: 0.7790107799619531 and parameters: {'C': 0.9413891050248867, 'penalty': 'l1'}. Best is trial 0 with value: 0.7790107799619531.
[I 2024-11-19 00:13:20,875] Trial 1 finished with value: 0.6668780384696682 and parameters: {'C': 0.007322732019395943, 'penalty': 'l2'}. Best is trial 0 with value: 0.7790107799619531.
[I 2024-11-19 00:13:21,356] Trial 2 finished with value: 0.7740435425914183 and parameters: {'C': 6.594319171916397, 'penalty': 'l2'}. Best is trial 0 with value: 0.7790107799619531.
[I 2024-11-19 00:13:21,450] Trial 3 finished with value: 0.5457619953498203 and parameters: {'C': 0.012355076134674167, 'penalty': 'l1'}. Best is trial 0 with value: 0.7790107799619531.
[I 2024-11-19 00:13:21,512] Trial 4 finished with value: 0.3333333333333333 and parameters: {'C': 0.0034365924574413847, 'penalty': 'l1'}. Best is 